In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import LeaveOneOut, StratifiedKFold, GridSearchCV
from sklearn.metrics import auc, roc_curve, average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
import shap

from func_prediction import sample_w_replacement

In [63]:
def classify_boostrap_inclSHAP2(X, 
                                y, 
                                pipe, 
                                hp_grid,
                                perc_samples_per_boostrap=1):
    
    ''' 
    Run classifier with bootstrapping and SHAP analysis
    '''


    ''' Bootstrap '''
    train_idx, test_idx = sample_w_replacement(X, 
                                               n_size=np.ceil(X.shape[0]*perc_samples_per_boostrap).astype("int"),
                                               stratify=y.to_numpy(),
                                               random_state=None)       

    print(train_idx, test_idx)

    X_train = X.iloc[train_idx].copy(); y_train = y.iloc[train_idx].copy()
    X_test = X.iloc[test_idx].copy(); y_test = y.iloc[test_idx].copy()

    ######### Bootstrapping sanity checks #########
    # _, a = np.unique(y_train,return_counts=True)
    # _, b = np.unique(y_test,return_counts=True)
    # print(f"Size X_train; labels y: {X_train.shape}, {a}")
    # print(f"Size X_test; labels y: {X_test.shape}, {b}")
    # print(train_idx, test_idx)
    ################################################

    ### Inner CV: random seed
    gs = GridSearchCV(pipe, hp_grid, scoring='balanced_accuracy', verbose=1, cv=3, n_jobs=-1) 
    gs.fit(X_train, y_train)

    ### Score
    y_predProba = gs.best_estimator_.predict_proba(X_test)[:,1]
    fpr, tpr, _ = roc_curve(y_test,y_predProba)
    auc = roc_auc_score(y_test, y_predProba)
    precision_recall = average_precision_score(y_test,y_predProba)
    dic_results = {"auc": auc,
		           "average_prec":precision_recall, 
                   "fpr":fpr, 
                   "tpr":tpr}  
    




    return X_test, test_idx, gs

In [69]:
def shappy(X, X_test, test_idx, gs):
    ''' 
    SHAP 
    '''
    # Use SHAP to explain predictions
    shap_values_per_bootstrap = dict()
    explainer = shap.TreeExplainer(gs.best_estimator_["classifier"])
    X_test_imputed = gs.best_estimator_["imputation"].transform(X_test)
    

    #print(explainer.shap_values(X_test_imputed)[0])
    shap_values = explainer.shap_values(X_test_imputed)[0] #[:,:,1]


    # Extract SHAP information per fold per sample 
    for i, idx in enumerate(test_idx):
        shap_values_per_bootstrap[idx] = shap_values[i] #-#-#

    ''' 
    Save pred_proba()
    '''    
    proba_per_bootstrap = dict()
    for i, idx in enumerate(X.iloc[test_idx]):
        proba_per_bootstrap[idx] = y_predProba[i] #-#-#    
    return proba_per_bootstrap

In [68]:

'''
Internal validation using bootstrapping (for CI calculation)

- boostrapping: resample with replacement (n_samples = X.shape[0])
- include hyperparameter tuning 
- n_iter ~ 200 [acc to this](https://sebastianraschka.com/blog/2022/confidence-intervals-for-ml.html#method-22-bootstrap-confidence-intervals-using-the-percentile-method)
- include SHAP analysis

(c) Sonja Katz, 2024
'''


PATH_base = "/home/WUR/katz001/PROJECTS/myaReg-genderDifferences"

import os
import numpy as np
import json
import pandas as pd
import sys 
import errno  
sys.path.append(f"{PATH_base}/scripts")
import seaborn as sns 
import matplotlib.pyplot as plt
import shap

from func_preprocess import read_data, subset_wo_missigness, remove_NA, parseVariables, clean_data, impute_scale 
from func_prediction import pipe_imputation_scaling, pipe_supervisedSelector


from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.pipeline import Pipeline
import pickle


''' 
Prepare data --> change here for different setups!
'''
n_bootstrap = 1

target = "gender"
percentBoruta = 80
saveFig_quickCheck = True

''' 
Select features
'''
vars = f"{target}_bootstrapped_iterativeBoruta_{percentBoruta}perc"
varPath = f"{PATH_base}/results/20_featureSelection/CV_v3_mgfaRecoded/{vars}.txt"

''' 
Define paths
'''
folderFigures = f"{PATH_base}/results/30_internalValidation/CV_v3_mgfaRecoded/{vars}"
os.makedirs(folderFigures, exist_ok=True)
resultsPath = f"{PATH_base}/results/30_internalValidation/CV_v3_mgfaRecoded/{vars}"
os.makedirs(resultsPath, exist_ok=True)


models = {
          'rfc': RandomForestClassifier(), 
         }               


grids = {'rfc':{
               'classifier__n_estimators': [100, 300, 700],     
               'classifier__max_depth': [2,4,6],         
               'classifier__max_features': [2,4,6],  
               }
         }   

''' 
Read data
'''
data = read_data(PATH_base,FILENAME="all_data_edited_v3_mgfaRecoded_inverse")
X_orig = data.drop(target, axis=1)
y_orig = data[target]

''' 
Split
'''
X = data.drop(target, axis=1)
y = data[target]

## FOR DEVELOPMENT PURPOSES: smaller dataset
X = X.iloc[:100,:]
y = y[:100]
print(X.shape)
print(y)

''' 
Read in variables
'''
sel_variables = pd.read_csv(varPath, header=None)[0].tolist()
print(sel_variables)

''' 
Prepare imputation and scaling
'''
num_columns = X.loc[:,sel_variables].select_dtypes(include=["float64"]).columns
bin_columns = X.loc[:,sel_variables].select_dtypes(include=["int64"]).columns
cat_columns = X.loc[:,sel_variables].select_dtypes(include=["object"]).columns
preprocessor = pipe_imputation_scaling(num_columns, bin_columns, cat_columns)  

''' 
Run Pipeline
'''
model = 'rfc'
dic_summary = dict()
dic_summary_shap = dict()
dic_summary_predProba = dict()


''' 
Assemble pipeline
'''
pipe = Pipeline([("selector", pipe_supervisedSelector(sel_variables)),
                    ("imputation", preprocessor),
                    ("classifier", models[model])])
X_test, idx, gs = classify_boostrap_inclSHAP2(X, 
                                                                    y, 
                                                                    pipe, 
                                                                    hp_grid=grids[model],
                                                                    perc_samples_per_boostrap=1)









LOADING DATA
(100, 40)
pid
P7X50M0V    1
0X13811D    1
002D7Y7Z    1
002G71A3    0
00C4EK1J    0
           ..
1FVVC4U3    0
1G92U7KD    0
1GL2EJND    0
1GU3XDJV    1
1HK9NGAM    1
Name: gender, Length: 100, dtype: Int64
['age_erstmanifestation', 'Diagnosedauer', 'scoreadl_neu', 'autoimmunerkrankungen_rbzu', 'achrak_rb', 'thymektomie_gr', 'mgfaklassifikation_schlimmste_historisch_rb']
[63 13 90  4 32 40 35 88 85 60 17 42 60 47 50 35 63 79 35 31 49 64  9 73
 97 32 90 50 79 21 63 95 97 97  4 49 55 32  3 40 88  4 67 91 86 71 58 10
 82  8 92 86 76 29 44 36 93 66 52 26 84 39 44 91 76 99 67 65 29  6 23  2
 15  8  0 33  8 12 67  2 25 80 87 33 80 80 84 57 10 15 75  0 93 65 87 65
 86 93 81 25] [ 5  7 16 22 24 27 28 30 37 41 53 56 62 68 69 74 96  1 11 14 18 19 20 34
 38 43 45 46 48 51 54 59 61 70 72 77 78 83 89 94 98]
Fitting 3 folds for each of 27 candidates, totalling 81 fits


In [70]:
shappy(X, X_test, idx, gs)

NameError: name 'y_predProba' is not defined

In [67]:
X

,zn_myasthener_krise_jn,zn_myasthener_exazerbation,aktueller_mgfa_score,aktuelle_MGFA_umkodiert,mgfaklassifikation_schlimmste_historisch_rb,okulaer,bulbaer,generalisiertemuskelschwaeche,muskelschmerz,autoimmunerkrankungen_rbzu,...,age,age_erstmanifestation,age_bei_diagnose,Diagnosedauer,seronegativ,scoreqmg_neu,scoreqol_neu,scoreadl_neu,chronicfatigue_neu,seelischesbefinden_neu
pid,,,,,,,,,,,,,,,,,,,,,
002G71A3,0,0,NaN,NaN,1,1,0,0,0,<NA>,...,64.0,64.0,64.0,0.0,0,6.0,NaN,NaN,NaN,NaN
00C4EK1J,0,0,1,1,3,1,1,0,0,0,...,82.0,NaN,73.0,NaN,0,NaN,NaN,NaN,NaN,NaN
01M51DX3,0,0,NaN,NaN,2,0,0,1,0,<NA>,...,39.0,16.0,NaN,NaN,0,7.0,20.0,0.0,NaN,NaN
072Y1KUZ,0,0,NaN,NaN,8,1,1,1,<NA>,0,...,54.0,51.0,51.0,0.0,0,NaN,25.0,3.0,NaN,NaN
0D31R9UX,0,1,3,3,5,1,1,0,0,0,...,82.0,80.0,80.0,0.0,0,5.0,16.0,NaN,23.0,11.0
0F9AFJ6P,0,0,NaN,NaN,3,1,1,1,0,0,...,52.0,52.0,52.0,0.0,0,NaN,6.0,2.0,16.0,10.0
0M3PGXUG,0,0,2,2,2,1,0,0,0,0,...,83.0,79.0,79.0,0.0,0,7.0,21.0,6.0,17.0,10.0
0MTZQL45,0,0,NaN,NaN,8,1,0,0,0,1,...,82.0,NaN,79.0,NaN,0,NaN,NaN,NaN,NaN,NaN
0WEZH3UF,0,0,0,0,3,1,1,1,0,0,...,71.0,69.0,69.0,0.0,0,NaN,5.0,0.0,14.0,7.0


In [ ]:
gs

GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('selector',
                                        <func_prediction.pipe_supervisedSelector object at 0x14eaee3cd3c0>),
                                       ('imputation',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('scaler',
                                                                                          MinMaxScaler()),
                                                                                         ('imputer',
                                                                                          IterativeImputer(n_nearest_features=5,
                                                                                                           sample_posterior=True))]),
                                                                         Index(['age_erstmanifestation', 'Diagnosedauer', 'scoreadl_ne...
                                                                                          OrdinalEncoder(dtype=<class 'numpy.int64'>,
                                                                                                         handle_unknown='use_encoded_value',
                                                                                                         unknown_value=9999))]),
                                                                         Index(['mgfaklassifikation_schlimmste_historisch_rb'], dtype='object'))])),
                                       ('classifier',
                                        RandomForestClassifier())]),
             n_jobs=-1,
             param_grid={'classifier__max_depth': [2, 4, 6],
                         'classifier__max_features': [2, 4, 6],
                         'classifier__n_estimators': [100, 300, 700]},
             scoring='balanced_accuracy', verbose=1)

In [35]:

### Save AUC and ROC for quick check
dic_summary[i] = dic_bootstrap_results
### Save SHAP values
for k,v in dic_shap_values.items():
    if k in dic_summary_shap.keys():
        dic_summary_shap[k].append(v)
    else:
        dic_summary_shap[k] = [v]
### Save individual predictions for further analyses
dic_summary_predProba[i] = dic_proba

NameError: name 'dic_bootstrap_results' is not defined